# JAYANTH AI — RAG + Ollama Project Notebook

**Goal:** Build a local PDF question-answering assistant using Python, Streamlit, ChromaDB, and Ollama.

**Time target:** 2 hours  
**Platform:** Windows 10/11  
**AI model:** `llama3.2:3b`  
**Embedding model:** `nomic-embed-text`

This notebook is a guided setup and project generator. Run the cells from top to bottom. Install Ollama, Python 3.12, and VS Code first if they are not already installed.

> **Important:** This notebook does not call a cloud AI API. Ollama runs the language and embedding models locally. The first model downloads require an internet connection.


## 1. Check your Python environment

Run this cell to see which Python version and operating system the notebook is using.

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Operating system:", platform.platform())


## 2. Install the Python packages

Run this cell once. If installation finishes but imports fail, restart the notebook kernel and continue.

In [ ]:
%pip install -U streamlit ollama chromadb pypdf


## 3. Check that Ollama is installed and running

Install Ollama for Windows from https://ollama.com/download/windows. Open PowerShell and run these commands if you have not downloaded the models yet:

```powershell
ollama pull llama3.2:3b
ollama pull nomic-embed-text
```

Then make sure the Ollama application is running. The next cell checks whether the local Ollama service responds.

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"
try:
    response = requests.get(OLLAMA_URL, timeout=5)
    print("Ollama service:", response.text)
except Exception as exc:
    print("Could not connect to Ollama.")
    print("Open the Ollama application, then run this cell again.")
    print("Details:", exc)


## 4. Verify that both models are available

If either model is missing, run the corresponding `ollama pull` command in PowerShell.

In [ ]:
import subprocess

for model in ["llama3.2:3b", "nomic-embed-text"]:
    result = subprocess.run(
        ["ollama", "show", model],
        capture_output=True, text=True
    )
    print(f"--- {model} ---")
    print("Available" if result.returncode == 0 else "Not found — run: ollama pull " + model)


## 5. Create the project folders

The application will be saved in a folder named `JAYANTH-RAG-OLLAMA` next to this notebook.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd() / "JAYANTH-RAG-OLLAMA"
DOCUMENTS_DIR = PROJECT_DIR / "documents"
DB_DIR = PROJECT_DIR / "chroma_db"

PROJECT_DIR.mkdir(exist_ok=True)
DOCUMENTS_DIR.mkdir(exist_ok=True)
DB_DIR.mkdir(exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Documents folder:", DOCUMENTS_DIR)


## 6. Generate the Streamlit RAG application

The app below:
1. Accepts PDF uploads.
2. Extracts text with `pypdf`.
3. Splits text into overlapping chunks.
4. Creates embeddings using Ollama's `nomic-embed-text`.
5. Stores chunks in a local ChromaDB collection.
6. Retrieves relevant chunks for each question.
7. Sends those chunks to `llama3.2:3b` to generate an answer.

The app includes a source section so you can inspect which PDF pages were retrieved.

In [ ]:
app_code = r"""
import hashlib
from pathlib import Path

import chromadb
import ollama
import streamlit as st
from pypdf import PdfReader

APP_DIR = Path(__file__).parent
DOCS_DIR = APP_DIR / "documents"
DB_DIR = APP_DIR / "chroma_db"
DOCS_DIR.mkdir(exist_ok=True)
DB_DIR.mkdir(exist_ok=True)

CHAT_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

st.set_page_config(page_title="JAYANTH AI | RAG Assistant", page_icon="🧠", layout="wide")
st.title("🧠 JAYANTH AI")
st.caption("Local PDF question-answering assistant powered by RAG + Ollama")

@st.cache_resource
def get_collection():
    client = chromadb.PersistentClient(path=str(DB_DIR))
    return client.get_or_create_collection(
        name="jayanth_documents",
        metadata={"hnsw:space": "cosine"}
    )

collection = get_collection()

def split_text(text, chunk_size=900, overlap=150):
    text = " ".join(text.split())
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

def embed(text):
    result = ollama.embeddings(model=EMBED_MODEL, prompt=text)
    return result["embedding"]

def index_pdf(uploaded_file):
    reader = PdfReader(uploaded_file)
    file_bytes = uploaded_file.getvalue()
    file_hash = hashlib.sha256(file_bytes).hexdigest()
    ids, docs, metas, vectors = [], [], [], []

    for page_num, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""
        for chunk_num, chunk in enumerate(split_text(page_text)):
            chunk_id = f"{file_hash}_{page_num}_{chunk_num}"
            ids.append(chunk_id)
            docs.append(chunk)
            metas.append({
                "filename": uploaded_file.name,
                "page": page_num,
                "chunk": chunk_num
            })
            vectors.append(embed(chunk))

    if not ids:
        return 0, "No extractable text found. This may be a scanned PDF; OCR is not included."

    # Upsert makes re-uploading the same PDF safe.
    collection.upsert(ids=ids, documents=docs, metadatas=metas, embeddings=vectors)
    return len(ids), None

with st.sidebar:
    st.header("📄 Add documents")
    uploads = st.file_uploader("Upload one or more text-based PDFs", type=["pdf"], accept_multiple_files=True)
    if st.button("Process uploaded PDFs", type="primary", disabled=not uploads):
        for pdf in uploads:
            with st.spinner(f"Reading and embedding {pdf.name}..."):
                try:
                    count, error = index_pdf(pdf)
                    if error:
                        st.warning(f"{pdf.name}: {error}")
                    else:
                        st.success(f"{pdf.name}: indexed {count} chunks.")
                except Exception as exc:
                    st.error(f"Could not process {pdf.name}: {exc}")

    st.divider()
    st.write(f"**Indexed chunks:** {collection.count()}")
    if st.button("Clear document index"):
        collection.delete(where={})
        st.session_state.messages = []
        st.rerun()

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        if message.get("sources"):
            with st.expander("Sources used"):
                for source in message["sources"]:
                    st.write(f"• {source}")

question = st.chat_input("Ask a question about your uploaded PDFs...")
if question:
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        if collection.count() == 0:
            answer = "Please upload and process at least one PDF first."
            st.markdown(answer)
            st.session_state.messages.append({"role": "assistant", "content": answer})
        else:
            with st.spinner("Searching your documents and generating an answer..."):
                try:
                    query_vector = embed(question)
                    results = collection.query(
                        query_embeddings=[query_vector],
                        n_results=min(4, collection.count()),
                        include=["documents", "metadatas", "distances"]
                    )
                    context_parts = []
                    sources = []
                    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
                        context_parts.append(doc)
                        source = f'{meta["filename"]}, page {meta["page"]}'
                        if source not in sources:
                            sources.append(source)

                    context = "\n\n---\n\n".join(context_parts)
                    prompt = f"""Answer the user's question using only the supplied document context.
If the context does not contain the answer, say that the uploaded documents do not provide enough information.
Do not invent facts. Be clear and concise.

DOCUMENT CONTEXT:
{context}

QUESTION:
{question}
"""
                    response = ollama.chat(
                        model=CHAT_MODEL,
                        messages=[{"role": "user", "content": prompt}]
                    )
                    answer = response["message"]["content"]
                    st.markdown(answer)
                    with st.expander("Sources used"):
                        for source in sources:
                            st.write(f"• {source}")
                    st.session_state.messages.append({
                        "role": "assistant", "content": answer, "sources": sources
                    })
                except Exception as exc:
                    answer = f"An error occurred: {exc}\n\nCheck that Ollama is running and both models are installed."
                    st.error(answer)
                    st.session_state.messages.append({"role": "assistant", "content": answer})
"""
app_path = PROJECT_DIR / "app.py"
app_path.write_text(app_code, encoding="utf-8")
print("Created:", app_path)


## 7. Create a requirements file

In [ ]:
requirements = """streamlit
ollama
chromadb
pypdf
"""
(PROJECT_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")
print((PROJECT_DIR / "requirements.txt").read_text())


## 8. Launch the chatbot

Run the command below in a **PowerShell terminal** (not in a Python cell). If your notebook is running inside VS Code, open Terminal → New Terminal.

```powershell
cd JAYANTH-RAG-OLLAMA
python -m streamlit run app.py
```

Streamlit should open the app in your browser, usually at `http://localhost:8501`.

If your terminal is in a different folder, use the full path printed in section 5.

**Important:** Keep Ollama running while using the chatbot.

## 9. Test the project

1. Upload a text-based PDF (for example, a class note or a short report).
2. Click **Process uploaded PDFs** and wait for embedding to finish.
3. Ask a question whose answer is present in that PDF.
4. Expand **Sources used** and check the filename and page numbers.
5. Ask a question that is not answered in the PDF. The assistant should say the documents do not provide enough information.

## 10. Troubleshooting

| Problem | What to do |
|---|---|
| `ollama` command not found | Install Ollama, restart PowerShell, and try again. |
| Cannot connect to localhost:11434 | Open the Ollama application and retry. |
| Model not found | Run `ollama pull llama3.2:3b` and `ollama pull nomic-embed-text`. |
| Streamlit command not found | Run `python -m pip install streamlit`, then use `python -m streamlit run app.py`. |
| PDF has no text | It may be scanned. This starter project does not include OCR. Try a text-based PDF. |
| Slow first response | The model may still be loading. First model runs can take longer. |

## 11. Project architecture

```text
PDF Upload
   |
   v
PyPDF text extraction
   |
   v
Chunking (900 characters, 150 overlap)
   |
   v
Ollama embedding model: nomic-embed-text
   |
   v
ChromaDB persistent vector store
   |
   v
User question -> question embedding
   |
   v
Retrieve top 4 relevant chunks
   |
   v
Ollama chat model: llama3.2:3b
   |
   v
Answer + source pages in Streamlit
```

## 12. What to explain during your project presentation

- **RAG (Retrieval-Augmented Generation):** A method that retrieves relevant information from external documents and gives it to a language model as context before generating an answer.
- **Ollama:** Runs language and embedding models locally.
- **Embeddings:** Numerical representations of text that help compare semantic similarity.
- **ChromaDB:** Stores document chunks and their embeddings and retrieves similar chunks.
- **Streamlit:** Provides the web interface.
- **Hallucination reduction:** The prompt instructs the model to answer from retrieved context and admit when the documents do not contain enough information. This reduces, but does not eliminate, incorrect answers.

**Limitations of this starter version:** It handles text-based PDFs, uses a simple character-based chunker, and has no user authentication or OCR. It is a local demo, not a production-secured service.
